# L09 Worked Solutions — Automation Agent Workflows

**Course:** AAU E26 — Introduction to Scripting, Data Mining and Machine Learning  
**Lecture:** Lecture 9 — Automation and agentic AI  
**Goal:** Practise safe tools, audit logs, stopping conditions, unexpected inputs, and human approval points.

[Open this notebook in Google Colab](https://colab.research.google.com/github/asmrabbi/E26_TAN7_Scripting_CPH/blob/main/notebooks/solutions/L09_solutions_automation_agent_workflows.ipynb) · [View the course repository](https://github.com/asmrabbi/E26_TAN7_Scripting_CPH)

Run the cells from top to bottom. Every executable line includes a short comment explaining what it does.


## Goal

Practise safe tools, audit logs, stopping conditions, unexpected inputs, and human approval points.


## Setup

A complete worked solution follows.


## Steps

1. Create one read-only tool that calculates a useful report fact.
2. Create a workflow that calls the tool and records an audit log.
3. Add a maximum-step condition and a human-approval condition.
4. Test normal, missing, and contradictory inputs.


In [1]:
def calculate_resolution_rate(report):  # Defines a read-only tool for one useful report fact.
    """Return a resolution proportion or None when required data is unsafe."""  # Documents the tool output.
    received = report.get("cases_received")  # Retrieves the denominator without changing the report.
    resolved = report.get("cases_resolved")  # Retrieves the numerator without changing the report.
    if received in (None, 0) or resolved is None:  # Checks for missing inputs and division by zero.
        return None  # Returns an explicit missing result instead of guessing.
    return resolved / received  # Calculates the rate for valid inputs.
def run_review_workflow(report, maximum_steps=2):  # Defines a bounded workflow around the read-only tool.
    """Log tool use and stop for human approval when the result is unsafe."""  # Documents the workflow controls.
    audit_log = []  # Creates an ordered record of actions and results.
    resolution_rate = calculate_resolution_rate(report)  # Calls the approved read-only tool.
    audit_log.append(f"calculate_resolution_rate -> {resolution_rate}")  # Records the tool result.
    if len(audit_log) >= maximum_steps:  # Checks the explicit maximum-step stopping condition.
        return {"status": "stopped at limit", "log": audit_log}  # Stops safely at the configured boundary.
    if resolution_rate is None or resolution_rate > 1:  # Checks for missing or contradictory results.
        return {"status": "human approval required", "log": audit_log}  # Stops before any external action.
    return {"status": "ready for approved next step", "log": audit_log}  # Reports readiness without performing an external action.
normal_report = {"record_id": 3001, "cases_received": 50, "cases_resolved": 45}  # Stores a complete plausible test case.
missing_report = {"record_id": 3002, "cases_received": 50, "cases_resolved": None}  # Stores a case with missing information.
contradictory_report = {"record_id": 3003, "cases_received": 50, "cases_resolved": 55}  # Stores a case that breaks the simplified count rule.
for test_report in [normal_report, missing_report, contradictory_report]:  # Runs the same workflow on all three cases.
    workflow_result = run_review_workflow(test_report)  # Collects the safe workflow result.
    print(test_report["record_id"], workflow_result)  # Displays each record identifier with its visible audit trail.


3001 {'status': 'ready for approved next step', 'log': ['calculate_resolution_rate -> 0.9']}
3002 {'status': 'human approval required', 'log': ['calculate_resolution_rate -> None']}
3003 {'status': 'human approval required', 'log': ['calculate_resolution_rate -> 1.1']}


## Checks

No message, upload, publication, or other external action should occur. Missing and contradictory reports must stop at human approval.


## Next Steps

Change one input, predict the result before running, and explain any difference between the expected and observed output.
